# Notebook 05 of 7 — What-If + Attribution + Paper

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

NB04 surfaced three names that look wrong. My instinct is to add to NVDA (it's up, so it's working, right?), trim VNQ (real estate has been dead), and close AMD (hostile calendar). Before I do any of that with real money — I'm going to diff the portfolio, decompose where my returns actually came from, and paper-trade the revised plan.

By the end of this notebook we will be able to answer one question:

> *What would my three intuitive trades actually do to my book, and did I decide them for the right reasons?*


### Provider chain for this notebook (Track A / #1433)

The same 5-tier chain from [NB01 §2](./01-getting-started-and-providers.ipynb):

> **fmp_cached → fmp → cboe → sec (EDGAR) → yfinance (last-resort, personal-use)**

NB05 is mostly *computation* over pickled state from NB01-NB04, but the
what-if / paper-trading cells fetch fresh quotes to mark positions:

| Data path used below | Primary | Free-authoritative fallback |
|---|---|---|
| Quote inputs for what-if / paper marks | `fmp_cached` | `cboe` `EquityQuote` (EOD) → yfinance (labeled) |
| Brinson-Fachler synthetic reference | computed | n/a |
| Position deltas / attribution | computed | n/a |

**No yfinance calls** in the shipped cells. Zero code cells change under
this PR — the note documents the fallback order so a reader knows what
happens when `fmp_cached` is stale.


In [ ]:
# [Phase B / NB05 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib
assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 1. Load state from NB03 + NB04

Basket from NB03. Events + smart-money artifacts from NB04. If they
don't exist (running NB05 standalone), regenerate the basket from the
locked list and stub the signals with a canned fallback so the diff
still runs.

*The code cell below loads all three artifacts.*

In [ ]:
# [Phase B / NB05 §1] Load state from NB01/NB03/NB04
# Any missing artifact triggers a fallback regeneration so the notebook
# runs standalone. Pickle safety documented per PR #1391 discipline.
import json
import pickle  # noqa: S403  # trusted local; state files under .notebook_state/, gitignored
from pathlib import Path

state = Path(".notebook_state")
BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12},
    {"symbol": "NVDA",  "weight": 0.10},
    {"symbol": "GOOGL", "weight": 0.08},
    {"symbol": "AAPL",  "weight": 0.08},
    {"symbol": "AMD",   "weight": 0.06},
    {"symbol": "QQQ",   "weight": 0.15},
    {"symbol": "VTI",   "weight": 0.20},
    {"symbol": "VNQ",   "weight": 0.08},
    {"symbol": "BND",   "weight": 0.10},
    {"symbol": "GLD",   "weight": 0.03},
]

# basket.json (NB01 output)
basket_path = state / "basket.json"
if basket_path.exists():
    basket = json.loads(basket_path.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {basket_path}")
else:
    basket = BASKET_LOCKED
    print(f"Regenerated basket from STORY_BIBLE locked list")

# xray.pkl (NB03)
xray_artifact = None
if (state / "xray.pkl").exists():
    xray_artifact = pickle.loads((state / "xray.pkl").read_bytes())  # noqa: S301
    print(f"Loaded xray.pkl — {len(xray_artifact.get('effective_positions', {}))} effective positions")
else:
    print("xray.pkl not found (skipping x-ray delta comparison)")

# smart_money.pkl (NB04)
sm_artifact = None
if (state / "smart_money.pkl").exists():
    sm_artifact = pickle.loads((state / "smart_money.pkl").read_bytes())  # noqa: S301
    n = len(sm_artifact.get("by_symbol", {}))
    warns = len(sm_artifact.get("warnings", []))
    print(f"Loaded smart_money.pkl — {n} scored names, {warns} warnings")
else:
    print("smart_money.pkl not found (falling back to hand-authored trades)")


Loaded basket from .notebook_state\basket.json
Loaded xray.pkl — 23 effective positions
Loaded smart_money.pkl — 3 scored names, 2 warnings


## 2. The three candidate trades — in English first

Before we touch the diff engine, write out what we intend to do in plain
words. Doing this by hand every time is a discipline; if I can't say
in one sentence why I'm placing the trade, I have no business placing
it.

The concept primer for sizing: the size of a trade matters at least
as much as its direction. The trader's problem is that *conviction*
and *risk of ruin* pull in opposite directions — the trade you're
most sure about is exactly the one you're tempted to over-size, and
that's how books blow up. The textbook answers are **Kelly** (bet
a fraction of bankroll proportional to edge/odds; mathematically
optimal for compounding but wildly aggressive in practice) and
**fixed-fraction** (risk a constant small % of book per trade, e.g.
1-2%; boring but survivable). The practitioner answer for retail
books is *Kelly-lite*: compute Kelly, then take a quarter or half of
it — you keep most of the compounding benefit and drop the
tail-of-ruin risk to something you can live with. Rules of thumb:
never let a single trade risk more than ~2% of book on stop-out;
never let a single name exceed the top-1 weight NB03 said you were
comfortable with; and — the discipline NB05 enforces below — every
paper order carries a mandatory rationale field, so you can't
retroactively invent the reason you took the trade.

Sam's three:

1. **Add to NVDA** — up on the year, positive smart-money in NB04.
2. **Trim VNQ** — REITs weak, rate outlook hostile.
3. **Close AMD** — earnings in 5 days, negative smart-money, holding
   into it feels wrong.

Write them down. We'll come back to this list.

> **📖 Position sizing** — deciding *how much* to allocate to each trade, given account size, conviction, and per-trade risk budget. Distinct from *asset allocation* (which sectors/classes get capital); sizing is the per-trade layer inside a given allocation. [Investopedia →](https://www.investopedia.com/terms/p/positionsizing.asp)
>
> **📖 Kelly criterion** — the fraction of bankroll that maximizes long-run compounded growth given a known edge and odds. Full-Kelly is theoretically optimal and practically insane; "Kelly-lite" (¼- to ½-Kelly) is how it's actually used. [Investopedia →](https://www.investopedia.com/terms/k/kellycriterion.asp)
>
> **📖 Risk management** — the umbrella discipline covering sizing, stops, diversification, and drawdown control. The rationale field on every paper order below is a risk-management artifact: it forces the trader to name the thesis so it can be falsified later. [Investopedia →](https://www.investopedia.com/terms/r/riskmanagement.asp)

*The code cell below encodes the three trades as a list of dicts with a
`rationale` field per trade — the same shape NB05 §6 will submit as
paper orders.*

In [ ]:
# [Phase B / NB05 §2] Three candidate trades in English first
# NB04 pickles top_conviction as plain dicts (JSON-safe), so we
# key-access rather than attribute-access here.  When NB04 falls back
# to the sanctioned fixture (STORY_BIBLE §3 item 3), we surface that
# label so Sam is honest with the reader about signal provenance.

def _get(item, key, default=None):
    if isinstance(item, dict):
        return item.get(key, default)
    return getattr(item, key, default)

top = (sm_artifact.get("top_conviction") if sm_artifact else None) or []
if top:
    label = sm_artifact.get("fixture_label") if sm_artifact.get("is_fixture") else None
    print("Using top_conviction from smart_money rollup"
          + (f" [{label}]" if label else " [live]") + ":")
    # Trim positive-composite names and close negative-composite names.
    trades = []
    for item in top[:3]:
        sym = _get(item, "symbol")
        comp = float(_get(item, "composite", 0.0))
        n = int(_get(item, "signal_count", 0))
        if comp > 0:
            action, delta = "buy", 5
        else:
            # Full close on negative-composite name; sizing is
            # illustrative — resolves against seeded lots below.
            action, delta = "close", -20
        trades.append({
            "symbol": sym,
            "action": action,
            "delta_shares": delta,
            "rationale": (
                f"smart_money composite {comp:+.2f} ({n} signals)"
                + (f" — {label}" if label else "")
            ),
        })
else:
    print("Falling back to hand-authored trades (smart_money returned empty):")
    trades = [
        {
            "symbol": "NVDA",
            "action": "buy",
            "delta_shares": 5,
            "rationale": "Add to NVDA — it's up on the year; my sub doesn't confirm 13F flow.",
        },
        {
            "symbol": "VNQ",
            "action": "trim",
            "delta_shares": -10,
            "rationale": "Trim VNQ — REITs weak, rate outlook hostile.",
        },
        {
            "symbol": "AMD",
            "action": "close",
            "delta_shares": -20,  # illustrative full close
            "rationale": "Close AMD — earnings in <7 days, don't want to hold into it.",
        },
    ]

print()
print(f"{'Symbol':<8}{'Action':<10}{'ΔShares':>10}   Rationale")
print("-" * 78)
for t in trades:
    print(f"{t['symbol']:<8}{t['action']:<10}{t['delta_shares']:>10}   {t['rationale']}")


Using top_conviction from smart_money rollup [example — signal shape as of 2026-06-30]:

Symbol  Action       ΔShares   Rationale
------------------------------------------------------------------------------
MSFT    buy                5   smart_money composite +0.72 (6 signals) — example — signal shape as of 2026-06-30
NVDA    buy                5   smart_money composite +0.61 (5 signals) — example — signal shape as of 2026-06-30
AMD     close            -20   smart_money composite -0.58 (4 signals) — example — signal shape as of 2026-06-30


## 3. What-if diff — PR #905

`obb.portfolio_intel` ships a stateless what-if engine (shipped in PR
#905). Give it a basket + a list of proposed trades, and it returns a
side-by-side before/after on every metric NB03 introduced: HHI, sector
weights, Sharpe, tracking error, MaxDD, top-K concentration,
single-name kill-shot.

If a trade looks *intuitively* like risk reduction but the diff
shows concentration going *up*, that's the tool catching you.

*The code cell below runs the what-if engine on Sam's three trades and
renders the before/after table with deltas highlighted.*

In [ ]:
# [Phase B / NB05 §3] What-if diff via run_whatif (from PR #905)
# The run_whatif engine needs full MarketData: prices, holdings map,
# covariance, benchmark returns. Building that from scratch is heavy;
# for the demo we do a MINIMAL diff in-notebook (weight-space) that
# mirrors what run_whatif produces for the "concentration + sector"
# columns. When Sam has the full ETF-Holdings sub-plan wired, they
# swap to run_whatif directly.
from decimal import Decimal

# Assume $100k paper account; convert current basket weights to
# nominal share counts using approximate prices from EquityQuote
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

ACCOUNT_SIZE = Decimal("100000")
prices = {}
for p in basket:
    try:
        q = obb.equity.price.quote(symbol=p["symbol"], provider="fmp_cached").to_df()
        price = float(q["last_price"].iloc[0])
        prices[p["symbol"]] = price
    except Exception:
        prices[p["symbol"]] = 100.0  # neutral fallback

# Current shares implied by weight × account_size / price
current_shares = {
    p["symbol"]: (float(ACCOUNT_SIZE) * p["weight"]) / prices[p["symbol"]]
    for p in basket
}

# Apply trade deltas
new_shares = dict(current_shares)
for t in trades:
    new_shares[t["symbol"]] = new_shares.get(t["symbol"], 0.0) + t["delta_shares"]

# Recompute weights
def _weights(shares_dict, price_dict) -> dict[str, float]:
    values = {s: shares_dict[s] * price_dict.get(s, 100.0) for s in shares_dict}
    total = sum(values.values())
    return {s: v/total for s, v in values.items() if total > 0}

w_current = _weights(current_shares, prices)
w_new = _weights(new_shares, prices)

# Delta table
print(f"{'Symbol':<8}{'Before':>10}{'After':>10}{'Δweight':>10}")
print("-" * 40)
for s in sorted(set(w_current) | set(w_new), key=lambda x: -abs(w_new.get(x, 0) - w_current.get(x, 0))):
    b, a = w_current.get(s, 0), w_new.get(s, 0)
    d = a - b
    if abs(d) < 0.001: continue
    print(f"{s:<8}{b*100:>9.2f}%{a*100:>9.2f}%{d*100:>+9.2f}%")

# HHI delta
def _hhi(w): return sum(x*x for x in w.values())
h_c, h_n = _hhi(w_current), _hhi(w_new)
neff_c = 1/h_c if h_c > 0 else float("nan")
neff_n = 1/h_n if h_n > 0 else float("nan")
print()
print(f"HHI:         {h_c:.4f} → {h_n:.4f}   Δ={h_n-h_c:+.4f}")
print(f"Effective-N: {neff_c:.2f}   → {neff_n:.2f}   Δ={neff_n-neff_c:+.2f}")


Symbol      Before     After   Δweight
----------------------------------------
AMD          6.00%    -4.80%   -10.80%
MSFT        12.00%    15.04%    +3.04%
NVDA        10.00%    11.93%    +1.93%
VTI         20.00%    21.62%    +1.62%
QQQ         15.00%    16.22%    +1.22%
BND         10.00%    10.81%    +0.81%
AAPL         8.00%     8.65%    +0.65%
GOOGL        8.00%     8.65%    +0.65%
VNQ          8.00%     8.65%    +0.65%
GLD          3.00%     3.24%    +0.24%

HHI:         0.1206 → 0.1474   Δ=+0.0268
Effective-N: 8.29   → 6.79   Δ=-1.51


## 4. Reading a what-if

Which deltas matter, which are noise:

- **HHI up + top-1 weight up** → you concentrated. That's usually
  wrong unless the smart-money signal on that name is overwhelming.
- **HHI up + top-1 weight down** → you spread the concentration across
  a few names. Sometimes intended, sometimes accidental.
- **Sharpe delta > +0.1** → suspicious on a small trade. What's
  giving you that much improvement from one trade?
- **MaxDD delta > +2%** → you added tail risk. Was that on purpose?

Sam's trades: the "add NVDA + trim VNQ" moves NB03's already-scary
Tech overweight *further up*. The what-if numbers should reflect
that. The "close AMD" helps sector diversity but the size is small so
the effect is muted.

*The code cell below prints a short interpretation for each row of the
diff table, in Sam's voice.*

In [ ]:
# [Phase B / NB05 §4] Reading a what-if — Sam-voice interpretations
delta_hhi = h_n - h_c
delta_neff = neff_n - neff_c

print("How to read this diff (Sam-voice):")
print()
if delta_hhi > 0.001:
    print(f"  HHI Δ = {delta_hhi:+.4f}  (concentration UP — my trades concentrated me)")
    print(f"  Effective-N Δ = {delta_neff:+.2f}  (fewer effective bets)")
    print(f"  → This is the tool catching me. The intuitive trade made things worse.")
elif delta_hhi < -0.001:
    print(f"  HHI Δ = {delta_hhi:+.4f}  (concentration DOWN — trades diversified me)")
    print(f"  Effective-N Δ = {delta_neff:+.2f}  (more effective bets)")
    print(f"  → Trades achieved the diversification I intended.")
else:
    print(f"  HHI barely moved ({delta_hhi:+.4f})")
    print(f"  → Trades were size-neutral; sector/name mix may still have shifted.")

# Individual trade contributions
print()
print("Per-trade contribution to portfolio-level delta:")
for t in trades:
    s = t["symbol"]
    before = w_current.get(s, 0)
    after = w_new.get(s, 0)
    print(f"  {s} {t['action']:<6}  weight {before*100:.2f}% → {after*100:.2f}%")


How to read this diff (Sam-voice):

  HHI Δ = +0.0268  (concentration UP — my trades concentrated me)
  Effective-N Δ = -1.51  (fewer effective bets)
  → This is the tool catching me. The intuitive trade made things worse.

Per-trade contribution to portfolio-level delta:
  MSFT buy     weight 12.00% → 15.04%
  NVDA buy     weight 10.00% → 11.93%
  AMD close   weight 6.00% → -4.80%


## 5. Brinson attribution — where did my returns come from?

The concept primer for attribution: at year-end you know your book beat
or lagged the benchmark by X%. The trader's problem is *why* —
because "I picked good stocks" and "I was in the right sectors at the
right time" are two very different skills, and confusing them means
you keep doing the thing that isn't actually working. **Brinson-Fachler
attribution** decomposes active return into three orthogonal
contributors: **allocation effect** (I was over/under-weight sectors
relative to the benchmark), **selection effect** (inside each sector,
I picked names that beat/lagged the sector), and **interaction**
(the cross-term, usually small enough to ignore for retail books).
Rules of thumb: for most retail books allocation dominates —
being overweight the year's winning sector matters more than
individual name picks; if your active return is meaningful and
allocation is the main contributor, you were making a sector-timing
bet whether you knew it or not; if selection is the dominant
contributor and it's negative, your stock-picking is destroying value
inside sectors you were correctly weighted in. What the platform adds
beyond the textbook decomposition: the same benchmark and sector
mapping used everywhere else in NB03/NB04, so the attribution numbers
line up with the concentration numbers.

Separate from the what-if: **why has my basket underperformed?** The
Brinson-Fachler decomposition against SPY splits my active return
into:

- **Allocation** — did I over/under-weight sectors that mattered?
- **Selection** — inside each sector, did I pick better or worse names
  than the benchmark?

The uncomfortable answer for most retail books: **allocation** drives
it, not **selection**. Being underweight tech in a tech-up year hurts
more than picking the "wrong" tech name. If my alpha is negative and
allocation is the dominant contributor, I was betting on
sector-timing without knowing it.

> **📖 Attribution analysis** — the decomposition of an active portfolio's return relative to a benchmark into its causes. Brinson-Fachler (§5 here) is one specific decomposition; there are factor-based variants (Fama-French, Carhart) that go further but need heavier machinery. [Investopedia →](https://www.investopedia.com/terms/a/attribution-analysis.asp)
>
> **📖 Asset allocation** — the top-level decision of what % goes to each asset class or sector. The allocation *effect* in Brinson measures whether your allocation drifts from the benchmark added or subtracted value. [Investopedia →](https://www.investopedia.com/terms/a/assetallocation.asp)
>
> **📖 Security selection** — the decision of *which specific securities* to hold within an allocation. The selection *effect* measures whether the names you picked inside each sector beat or lagged the sector average. [Investopedia →](https://www.investopedia.com/terms/s/security-selection.asp)

*The code cell below runs Brinson-Fachler on the basket vs SPY over
the last 12 months and prints the allocation-vs-selection decomposition
by sector.*

In [ ]:
# [Phase B / NB05 §5] Brinson attribution — allocation vs selection
# Uses openbb_portfolio_intel.analytics.brinson.oracle.brinson_reference,
# a reference implementation used in the analytics test suite. Input
# is a per-sector DataFrame with portfolio weight, benchmark weight,
# portfolio return, benchmark return.
#
# For the demo we build a small hand-authored per-sector table.
# In production Sam would feed real per-position returns aggregated
# by sector vs SPY sector weights.
import pandas as pd
from openbb_portfolio_intel.analytics.brinson.oracle import brinson_reference

df = pd.DataFrame([
    # (sector, portfolio wt, benchmark wt, portfolio ret, benchmark ret)
    {"sector":"Technology",    "w_p":0.50, "w_b":0.30, "r_p": 0.08, "r_b": 0.10},
    {"sector":"Financials",    "w_p":0.05, "w_b":0.15, "r_p": 0.02, "r_b": 0.05},
    {"sector":"Healthcare",    "w_p":0.05, "w_b":0.15, "r_p": 0.03, "r_b": 0.02},
    {"sector":"Consumer",      "w_p":0.10, "w_b":0.15, "r_p": 0.04, "r_b": 0.06},
    {"sector":"Real Estate",   "w_p":0.10, "w_b":0.05, "r_p":-0.02, "r_b":-0.01},
    {"sector":"Bond Fund",     "w_p":0.10, "w_b":0.10, "r_p": 0.01, "r_b": 0.02},
    {"sector":"Commodity",     "w_p":0.10, "w_b":0.10, "r_p": 0.05, "r_b": 0.03},
])

effects = brinson_reference(df)
print("Brinson-Fachler attribution (portfolio vs benchmark):")
print(f"  active_return:       {effects.active_return:+.4f}")
print(f"  allocation:          {effects.allocation:+.4f}")
print(f"  selection:           {effects.selection:+.4f}")
print(f"  interaction:         {effects.interaction:+.4f}")
print()
print("Reconciliation check (should hold within rounding):")
sum_effects = effects.allocation + effects.selection + effects.interaction
print(f"  alloc + select + interact = {sum_effects:+.4f}  (vs active_return {effects.active_return:+.4f})")
print()
print("How to read (Sam-voice):")
print("  |allocation| > |selection|  → my active return came from sector-timing")
print("  |selection| > |allocation|  → my active return came from stock-picking")
print("  Both small (near 0)         → I'm close to the benchmark; no active bet paid off")
print()
print("For this synthetic table:")
if abs(effects.allocation) > abs(effects.selection):
    print(f"  → allocation dominates ({effects.allocation:+.4f} vs {effects.selection:+.4f})")
    print(f"    Sector-timing drove the active return, not stock-picking.")
else:
    print(f"  → selection dominates ({effects.selection:+.4f} vs {effects.allocation:+.4f})")
    print(f"    Stock-picking drove the active return, not sector-timing.")


Brinson-Fachler attribution (portfolio vs benchmark):
  active_return:       -0.0035
  allocation:          +0.0095
  selection:           -0.0115
  interaction:         -0.0015

Reconciliation check (should hold within rounding):
  alloc + select + interact = -0.0035  (vs active_return -0.0035)

How to read (Sam-voice):
  |allocation| > |selection|  → my active return came from sector-timing
  |selection| > |allocation|  → my active return came from stock-picking
  Both small (near 0)         → I'm close to the benchmark; no active bet paid off

For this synthetic table:
  → selection dominates (-0.0115 vs +0.0095)
    Stock-picking drove the active return, not sector-timing.


## 6. Paper blotter — placing the trades without capital

The concept primer for paper trading: paper is the rehearsal space
between "I have a thesis" and "I have live capital at risk." The
trader's problem is that ideas that look great on a spreadsheet often
fall apart the moment execution meets a real order book — fills
happen at the wrong price, orders sit unfilled, alerts don't fire when
you expected. Rules of thumb: run any new strategy on paper for at
least a full cycle of the thing you're trying to trade (a full
earnings season for an earnings play, a full month for a monthly
rebalance) before touching real capital; if the paper book underperforms
its own thesis, do NOT deploy it live "because live psychology will fix
it"; every paper order carries the same discipline as a real one
(rationale, size, stop, target). What the platform adds beyond a
textbook paper-trading platform: the **mandatory rationale field**
means you cannot look back six months later and pretend you knew what
you were doing — the entry thesis is captured with the fill.

`openbb_portfolio_intel/paper/` is the paper-trading engine. Every
paper order carries a **mandatory rationale field** — a discipline the
system enforces to make sure I can never claim later "I don't remember
why I bought this." NB07's Monday-morning routine picks up the blotter
state.

> **📖 Paper trade** — a trade recorded and marked as if live, but with no capital at risk. Sometimes called "simulated trading." The value is in exposing thesis and execution flaws before they cost money. [Investopedia →](https://www.investopedia.com/terms/p/papertrade.asp)
>
> **📖 Market order** — an instruction to buy or sell at the best currently available price. Fills fast, price uncertain. Fine for liquid mega-caps in the middle of the trading day; dangerous at the open, close, or on thin names. [Investopedia →](https://www.investopedia.com/terms/m/marketorder.asp)
>
> **📖 Limit order** — an instruction to buy or sell only at a specified price or better. Price certain, fill uncertain. The default order type for anyone who cares about slippage. [Investopedia →](https://www.investopedia.com/terms/l/limitorder.asp)
>
> **📖 Time in force (TIF)** — the instruction telling the broker how long an unfilled order stays live. Common values: **DAY** (cancels at close today), **GTC** (good-'til-canceled — stays live until filled or manually canceled, subject to broker maximum, usually 60-90 days). Getting this wrong is how stops silently disappear overnight. [Investopedia →](https://www.investopedia.com/terms/t/timeinforce.asp)
>
> **📖 Good-'til-canceled (GTC)** — the specific TIF that persists across sessions. The mandatory choice for any protective stop; the wrong choice for any thesis-timing entry. [Investopedia →](https://www.investopedia.com/terms/g/gtc.asp)
>
> **📖 Realized P&L** — profit or loss on positions you've actually closed, i.e. sold. This is the number the IRS cares about. [Investopedia →](https://www.investopedia.com/terms/r/realizedprofit.asp)
>
> **📖 Unrealized P&L** — profit or loss on positions still open, marked at current price. Also called "paper gains/losses" (unrelated to paper trading). The number moves with the market; it's not tax-triggering until you close. [Investopedia →](https://www.investopedia.com/terms/u/unrealizedgain.asp)

*The code cell below submits Sam's three trades to the paper blotter
with their rationales, marks the fills at current mid-quote, and
renders the blotter state.*

In [ ]:
# [Phase B / NB05 §6] Paper trade blotter — submit 3 trades, mark, render
# Uses InMemory paper engine (no persistence beyond notebook session).
# submit_order needs: OrderRequest, user_id, account_id, stores, quote_fetcher.
from decimal import Decimal
from datetime import datetime, timezone
from openbb_portfolio_intel.paper.accounts import (
    AccountConfig, InMemoryAccountStore,
)
from openbb_portfolio_intel.paper.fills import (
    OrderRequest, OrderType, TimeInForce, Quote,
    InMemoryPositionStore, submit_order, OrderRejected,
)

account_store = InMemoryAccountStore()
position_store = InMemoryPositionStore()

# Create paper account via the store's factory (which handles created_at etc.)
now = datetime.now(timezone.utc)
cfg = AccountConfig(starting_cash=Decimal("100000"), display_name="Sam-NB05")
account = account_store.create(
    user_id="sam",
    config=cfg,
    now=now,
    account_id="nb05-demo",
)

# Quote fetcher — Protocol requires .fetch(symbol, *, now) → Quote
class NotebookQuoteFetcher:
    def fetch(self, symbol: str, *, now):
        return Quote(
            symbol=symbol,
            last=Decimal(str(prices.get(symbol, 100.0))),
            quoted_at=now,
        )

quote_fetcher = NotebookQuoteFetcher()

# Seed the paper book with pre-existing positions in VNQ + AMD so the
# trim/close trades from CELL_TRADES resolve against real inventory
# (a fresh blotter can't sell what it doesn't own). This mirrors the
# reality of the NB03 basket that Sam is already holding.
from openbb_portfolio_intel.paper.fills import Lot
seed_lots = [
    Lot(symbol="VNQ", qty=Decimal("40"), avg_cost=Decimal("85"), realized_pnl=Decimal("0")),
    Lot(symbol="AMD", qty=Decimal("30"), avg_cost=Decimal("135"), realized_pnl=Decimal("0")),
]
for lot in seed_lots:
    position_store.put("nb05-demo", lot, user_id="sam")

results = []
for t in trades:
    req = OrderRequest(
        symbol=t["symbol"],
        qty=Decimal(str(t["delta_shares"])),
        order_type=OrderType.MARKET,
        time_in_force=TimeInForce.DAY,
    )
    try:
        r = submit_order(
            req=req,
            user_id="sam",
            account_id="nb05-demo",
            account_store=account_store,
            position_store=position_store,
            quote_fetcher=quote_fetcher,
            now=now,
        )
        results.append((t, r))
    except OrderRejected as exc:
        results.append((t, ("REJECTED", str(exc))))

print("Paper blotter — 3 trades submitted at market:")
print(f"  {'Symbol':<8}{'Action':<10}{'ΔShares':>10}{'Fill':>14}{'Status':>12}")
print(f"  {'-'*8}{'-'*10}{'-'*10}{'-'*14}{'-'*12}")
for t, r in results:
    if isinstance(r, tuple) and r[0] == "REJECTED":
        print(f"  {t['symbol']:<8}{t['action']:<10}{t['delta_shares']:>10}{'—':>14}  REJECTED  {r[1][:40]}")
    else:
        fill = getattr(r, "fill", None) or (r.fills[0] if getattr(r, "fills", None) else None)
        status = getattr(r, "status", "?")
        price = getattr(fill, "price", "?") if fill else "?"
        price_str = f"${float(price):,.2f}" if isinstance(price, Decimal) else str(price)
        print(f"  {t['symbol']:<8}{t['action']:<10}{t['delta_shares']:>10}{price_str:>14}{str(status):>12}")

# Post-trade cash
post_account = account_store.get("nb05-demo", user_id="sam")
print(f"\nPost-trade cash: ${post_account.cash_balance:,.2f} (started at $100,000)")

# Post-trade positions — position_store only has .get(account_id, symbol)
print("\nPost-trade positions (querying store per trade symbol):")
seen_syms = {t["symbol"] for t in trades}
positions_snapshot = {}
for sym in seen_syms:
    try:
        pos = position_store.get("nb05-demo", sym, user_id="sam")
        if pos is not None:
            qty = getattr(pos, "qty", None) or getattr(pos, "quantity", None)
            positions_snapshot[sym] = float(qty) if qty is not None else 0.0
            print(f"  {sym}: qty={qty}")
    except Exception as exc:
        print(f"  {sym}: (store.get raised {type(exc).__name__})")


Paper blotter — 3 trades submitted at market:
  Symbol  Action       ΔShares          Fill      Status
  ------------------------------------------------------
  MSFT    buy                5       $381.89OrderStatus.FILLED
  NVDA    buy                5       $206.94OrderStatus.FILLED
  AMD     close            -20       $521.69OrderStatus.FILLED

Post-trade cash: $107,489.61 (started at $100,000)

Post-trade positions (querying store per trade symbol):
  NVDA: qty=5
  MSFT: qty=5
  AMD: qty=10


## 7. Low-BP alert — deliberately trigger one

The paper engine emits `Alert` objects on portfolio conditions: a
low-buying-power state, a GTC order about to expire, a stop level
approaching. To see the shape without waiting for a real event, we
submit an over-sized order that will not fit under simulated buying
power.

> **📖 Buying power** — the total dollar amount of securities the account can currently purchase. For a cash account it equals settled cash; for a margin account it's cash plus available margin. Falling below the threshold your alert is set to fires the low-BP alert below. [Investopedia →](https://www.investopedia.com/terms/b/buyingpower.asp)
>
> **📖 Margin** — borrowing from the broker against portfolio value to buy more securities. Levers returns *and* losses; a low-BP condition on a margin account is usually the early warning for a margin call. Paper mode here is single-account no-margin, so the alert is a pure sizing check — but the same alert plumbing is what would carry margin warnings in a live adapter. [Investopedia →](https://www.investopedia.com/terms/m/margin.asp)

*The code cell below submits an over-sized order, catches the alert,
and prints its full `Alert` shape.*

In [ ]:
# [Phase B / NB05 §7] Low-buying-power alert — deliberately trigger one
# Submit an over-sized order that would blow through remaining cash.
# The engine should reject with OrderRejected (or the equivalent) and
# we treat that rejection as an "Alert" — the shape a real alerts
# pipeline would surface.
from decimal import Decimal
from openbb_portfolio_intel.paper.fills import (
    OrderRequest, OrderRejected, OrderType, OrderStatus,
)

huge = OrderRequest(
    symbol="MSFT",
    qty=Decimal("10000"),  # ~$3.8M of MSFT on a $100k account
    order_type=OrderType.MARKET,
)
alert_fired = False
try:
    r = submit_order(
        req=huge,
        user_id="sam",
        account_id="nb05-demo",
        account_store=account_store,
        position_store=position_store,
        quote_fetcher=quote_fetcher,
        now=now,
    )
    # Engine may return a SubmitResult with status=REJECTED rather than raise.
    if getattr(r, "status", None) == OrderStatus.REJECTED:
        alert_fired = True
        reason = getattr(r, "reason", "unknown")
        print("Alert fired — SubmitResult(status=REJECTED):")
        print(f"  reason: {reason}")
    else:
        print(f"UNEXPECTED: over-sized order not rejected — result: {r}")
except OrderRejected as exc:
    alert_fired = True
    print("Alert fired — OrderRejected raised:")
    print(f"  reason: {exc}")
except Exception as exc:
    alert_fired = True
    print(f"Alert fired (different exception type): {type(exc).__name__}: {exc}")

if alert_fired:
    print()
    print("This is the shape a real Alert object would carry:")
    print(f"  Alert(type='low_buying_power', symbol='MSFT', requested_qty=10000,")
    print(f"        cash_available=${post_account.cash_balance:,.2f})")


Alert fired — SubmitResult(status=REJECTED):
  reason: insufficient cash: cash delta -3818908.50000 would drive 'nb05-demo' balance to -3711418.890850, below min_balance=0

This is the shape a real Alert object would carry:
  Alert(type='low_buying_power', symbol='MSFT', requested_qty=10000,
        cash_available=$107,489.61)


## 8. Save state for NB07

`.notebook_state/paper_blotter.pkl` — the blotter + the alert object.
NB07's Monday-morning routine reloads it.

*The code cell below pickles the blotter state.*

In [ ]:
# [Phase B / NB05 §8] Save paper-blotter state for NB07
# Pickle safety: trusted-local-only, gitignored, never shipped.
import pickle  # noqa: S403
from pathlib import Path

state = Path(".notebook_state")

# Serialize what NB07 needs — trades submitted + final cash + positions
blotter_artifact = {
    "trades_submitted": [t for t, _ in results],
    "starting_cash": 100000.0,
    "ending_cash": float(post_account.cash_balance),
    "positions_post": positions_snapshot,
    "whatif_diff": {
        "hhi_before": h_c, "hhi_after": h_n, "hhi_delta": h_n - h_c,
        "neff_before": neff_c, "neff_after": neff_n, "neff_delta": neff_n - neff_c,
    },
    "brinson_effects": {
        "active_return": float(effects.active_return),
        "allocation":    float(effects.allocation),
        "selection":     float(effects.selection),
        "interaction":   float(effects.interaction),
    },
}

out = state / "paper_blotter.pkl"
out.write_bytes(pickle.dumps(blotter_artifact))
print(f"Wrote (repo-rel): {str(out):<40}  {out.stat().st_size:,} bytes")
print()
print("NB07 loads this to render the paper-trading blotter in the reunion chapter.")


Wrote (repo-rel): .notebook_state\paper_blotter.pkl         785 bytes

NB07 loads this to render the paper-trading blotter in the reunion chapter.


---

## What is NOT in this notebook

- **Live broker adapter.** Paper only. The `execution/` module has the shape a real broker adapter would slot into; not shipped.
- **Margin-call simulation.** The buying-power model is single-account, no margin math yet.
- **Overnight risk report.** The `Alert` engine covers today; overnight-gap risk projection is future work.

## Preview of NB06

The paper trade is on. But NB05 was one moment — one basket, one set of signals, one diff. The strategy behind it — 'rebalance to top-5 owner-earnings yield inside the basket every month' — hasn't been tested against history. In NB06 I turn it into a `BacktestConfig` and see whether the underlying idea has real edge, or whether NB05 was just one lucky what-if.


## 📚 Further reading

Every Investopedia link cited in this notebook:

- [Position sizing — Investopedia](https://www.investopedia.com/terms/p/positionsizing.asp)
- [Kelly criterion — Investopedia](https://www.investopedia.com/terms/k/kellycriterion.asp)
- [Risk management — Investopedia](https://www.investopedia.com/terms/r/riskmanagement.asp)
- [Attribution analysis — Investopedia](https://www.investopedia.com/terms/a/attribution-analysis.asp)
- [Asset allocation — Investopedia](https://www.investopedia.com/terms/a/assetallocation.asp)
- [Security selection — Investopedia](https://www.investopedia.com/terms/s/security-selection.asp)
- [Paper trade — Investopedia](https://www.investopedia.com/terms/p/papertrade.asp)
- [Market order — Investopedia](https://www.investopedia.com/terms/m/marketorder.asp)
- [Limit order — Investopedia](https://www.investopedia.com/terms/l/limitorder.asp)
- [Time in force — Investopedia](https://www.investopedia.com/terms/t/timeinforce.asp)
- [Good-'til-canceled — Investopedia](https://www.investopedia.com/terms/g/gtc.asp)
- [Realized profit — Investopedia](https://www.investopedia.com/terms/r/realizedprofit.asp)
- [Unrealized gain — Investopedia](https://www.investopedia.com/terms/u/unrealizedgain.asp)
- [Buying power — Investopedia](https://www.investopedia.com/terms/b/buyingpower.asp)
- [Margin — Investopedia](https://www.investopedia.com/terms/m/margin.asp)

**Canonical references beyond Investopedia:**

- Brinson, G. P., Hood, L. R. & Beebower, G. L. — "Determinants of
  Portfolio Performance," *Financial Analysts Journal* 42(4), 1986. The
  original attribution decomposition; the Brinson-Fachler variant used
  in §5 extends it by benchmarking the allocation effect against the
  sector's excess return.
- Thorp, E. O. — "The Kelly Criterion in Blackjack, Sports Betting, and
  the Stock Market," *Handbook of Asset and Liability Management*,
  vol. 1, 2006. The practitioner's tour of Kelly sizing including the
  arguments for fractional-Kelly ("Kelly-lite") in real-world portfolios.


## 📚 Further reading

Every Investopedia link cited in this notebook:

- [Position sizing — Investopedia](https://www.investopedia.com/terms/p/positionsizing.asp)
- [Kelly criterion — Investopedia](https://www.investopedia.com/terms/k/kellycriterion.asp)
- [Risk management — Investopedia](https://www.investopedia.com/terms/r/riskmanagement.asp)
- [Attribution analysis — Investopedia](https://www.investopedia.com/terms/a/attribution-analysis.asp)
- [Asset allocation — Investopedia](https://www.investopedia.com/terms/a/assetallocation.asp)
- [Security selection — Investopedia](https://www.investopedia.com/terms/s/security-selection.asp)
- [Paper trade — Investopedia](https://www.investopedia.com/terms/p/papertrade.asp)
- [Market order — Investopedia](https://www.investopedia.com/terms/m/marketorder.asp)
- [Limit order — Investopedia](https://www.investopedia.com/terms/l/limitorder.asp)
- [Time in force — Investopedia](https://www.investopedia.com/terms/t/timeinforce.asp)
- [Good-'til-canceled — Investopedia](https://www.investopedia.com/terms/g/gtc.asp)
- [Realized profit — Investopedia](https://www.investopedia.com/terms/r/realizedprofit.asp)
- [Unrealized gain — Investopedia](https://www.investopedia.com/terms/u/unrealizedgain.asp)
- [Buying power — Investopedia](https://www.investopedia.com/terms/b/buyingpower.asp)
- [Margin — Investopedia](https://www.investopedia.com/terms/m/margin.asp)

**Canonical references beyond Investopedia:**

- Brinson, G. P., Hood, L. R. & Beebower, G. L. — "Determinants of
  Portfolio Performance," *Financial Analysts Journal* 42(4), 1986. The
  original attribution decomposition; the Brinson-Fachler variant used
  in §5 extends it by benchmarking the allocation effect against the
  sector's excess return.
- Thorp, E. O. — "The Kelly Criterion in Blackjack, Sports Betting, and
  the Stock Market," *Handbook of Asset and Liability Management*,
  vol. 1, 2006. The practitioner's tour of Kelly sizing including the
  arguments for fractional-Kelly ("Kelly-lite") in real-world portfolios.
